# Conv1D → LSTM → softmax: treino com walk-forward e backtest com custos

## Terceiro notebook da sequência

| Notebook | Entrega |
|---|---|
| `coleta_dados_b3.ipynb` | `data/raw/b3_cotacoes.parquet` — OHLCV diário |
| `preparacao_dados_cnn_lstm.ipynb` | `tensores/fold_<ano>.npz` — tensores $(N, T, F)$ já normalizados e rotulados |
| **este** | o modelo, o treino walk-forward e a conversão de acerto em **dinheiro** |

Nada aqui recalcula features ou rótulos. Se você precisar mexer neles, mexa **no notebook
anterior** e gere os tensores de novo — é o que separa um pipeline de uma pilha de células.

### O caminho desta aula

| Fase | O que fazemos |
|------|---------------|
| **0** | Carregar os folds e conferir o que veio |
| **1** | Por que **Conv1D + LSTM** para série temporal (e a armadilha do formato) |
| **2** | O modelo em PyTorch, com rastreio de formatos |
| **3** | Treinar **um** fold: pesos de classe, early stopping, curvas |
| **4** | Avaliar contra baselines que *não* são fáceis de bater |
| **5** | **Walk-forward**: 7 modelos, 7 anos fora da amostra |
| **6** | **Backtest com custos**: emolumentos, slippage, IR e giro |
| **7** | O diagnóstico honesto: acurácia *versus* lucro |

### O papel simulado

Tudo aqui é **PETR4** — e este notebook **não escolhe** isso. O de preparação construiu features e
rótulos a partir de um papel e gravou qual foi dentro dos próprios tensores; a Fase 0 lê esse
campo e a Fase 6 usa o mesmo valor no backtest.

Isso fecha uma porta específica: aplicar o sinal de um ativo ao preço de **outro**. Os tensores de
`VALE3` têm exatamente o mesmo formato dos de `PETR4`, as datas casam, o código roda até o fim — e
o resultado é ficção. Não há exceção para levantar, então a proteção tem que ser o **dado carregar
a própria identidade**. Para trocar de papel, troque no notebook de preparação e gere os tensores
de novo.

### O aviso que precisa vir antes do código

Este notebook termina em um número de retorno financeiro. Ele **não é** recomendação de
investimento. E o desfecho não é nenhum dos dois que se costuma ver: a rede **encontra** uma
vantagem estatística real, com resultado positivo até por operação — e ainda assim o resultado
final **perde para comprar e segurar, e perde para a Selic**. A parte didática está inteira em
descobrir **onde**, entre uma coisa e outra, a vantagem se dissolveu.

---

---

# FASE 0 — AMBIENTE E CARGA DOS FOLDS

In [ ]:
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, TensorDataset

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (12, 4)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# CPU basta: o modelo tem dezenas de milhares de parametros, nao dezenas de milhoes.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch  :", torch.__version__)
print("device :", device)

CLASSES = {0: "venda", 1: "mantém", 2: "compra"}
NOMES = [CLASSES[i] for i in range(3)]
DIR_TENSORES = Path("tensores")

In [ ]:
def carregar_folds(diretorio=DIR_TENSORES) -> dict:
    """Le os fold_<ano>.npz gerados pelo notebook de preparacao."""
    arquivos = sorted(diretorio.glob("fold_*.npz"))
    if not arquivos:
        raise FileNotFoundError(
            f"nenhum fold_*.npz em {diretorio.resolve()}. "
            "Rode preparacao_dados_cnn_lstm.ipynb antes deste notebook.")
    folds = {}
    for caminho in arquivos:
        ano = int(caminho.stem.split("_")[1])
        d = np.load(caminho, allow_pickle=True)
        folds[ano] = {k: d[k] for k in d.files}
    return folds


def ler_procedencia(folds: dict) -> dict:
    """Extrai ticker/fonte/rotulagem dos tensores e exige que todos concordem.

    Sem esses campos o .npz e anonimo: nada impede aplicar o sinal de um papel ao
    preco de outro -- erro que nao levanta excecao nenhuma. Tensores gerados por uma
    versao antiga do notebook de preparacao nao os tem; avisamos e usamos o valor
    declarado na Fase 6, que passa a ser responsabilidade de quem roda.
    """
    campos = {}
    for chave in ("ticker", "fonte", "rotulagem"):
        valores = {str(f[chave]) for f in folds.values() if chave in f}
        if not valores:
            campos[chave] = None
        elif len(valores) > 1:
            raise ValueError(f"folds discordam em '{chave}': {valores} -- "
                             "tensores de execucoes diferentes misturados na pasta")
        else:
            campos[chave] = valores.pop()
    return campos


folds = carregar_folds()
FEATURES = [str(f) for f in folds[min(folds)]["features"]]
T, F = folds[min(folds)]["X_train"].shape[1:]
PROC = ler_procedencia(folds)

print(f"{len(folds)} folds: {sorted(folds)}")
print(f"janela T = {T} dias | features F = {len(FEATURES)}")
print(f"papel      : {PROC['ticker'] or '?? (tensor sem procedência)'}")
print(f"fonte      : {PROC['fonte'] or '?? (tensor sem procedência)'}")
print(f"rotulagem  : {PROC['rotulagem'] or '??'}")

if PROC["ticker"] is None:
    print("\n[aviso] estes tensores foram gerados por uma versão do notebook de preparação")
    print("        que não gravava procedência. Rode-o de novo para ativar a checagem da Fase 6.")

linhas = []
for ano, f in sorted(folds.items()):
    linhas.append({
        "ano_teste": ano,
        "n_treino": len(f["y_train"]), "n_val": len(f["y_val"]), "n_teste": len(f["y_test"]),
        "teste_de": str(f["dates_test"][0]), "teste_até": str(f["dates_test"][-1]),
        "pesos": np.round(f["class_weights"], 2).tolist(),
    })
pd.DataFrame(linhas).set_index("ano_teste")

In [ ]:
# O porteiro de novo. O notebook anterior já garante isto, mas quem consome um
# artefato não confia na promessa de quem o produziu -- confere. Custou caro aprender:
# um único inf na preparação transformou uma coluna inteira em NaN nos tensores.
problemas = []
for ano, f in folds.items():
    for parte in ("X_train", "X_val", "X_test"):
        X = f[parte]
        if not np.isfinite(X).all():
            problemas.append(f"{ano}/{parte}: {int((~np.isfinite(X)).sum())} valores não finitos")
    for parte in ("y_train", "y_val", "y_test"):
        y = f[parte]
        if len(y) and not set(np.unique(y)).issubset({0, 1, 2}):
            problemas.append(f"{ano}/{parte}: rótulos fora de {{0,1,2}}")

print("\n".join(problemas) if problemas else "todos os folds finitos, rótulos em {0,1,2}")
assert not problemas

# Distribuição das classes no teste de cada ano -- é o que o modelo vai enfrentar
dist = pd.DataFrame({
    ano: pd.Series(f["y_test"]).value_counts(normalize=True).reindex([0, 1, 2]).fillna(0)
    for ano, f in sorted(folds.items())
}).rename(index=CLASSES).round(3)
print("\ndistribuição das classes no conjunto de TESTE, por ano:")
dist

### 🔎 Leia essa tabela antes de escrever o modelo

Duas informações que já limitam o que é possível:

**1. O conjunto de treino é minúsculo.** Algumas centenas de janelas no primeiro fold, pouco
mais de duas mil no último. Redes neurais são famintas; esta vai ter mais parâmetros do que
amostras se não tomarmos cuidado. Guarde o número — vamos comparar com a contagem de
parâmetros na Fase 2.

**2. A distribuição de classes muda de ano para ano.** Não é defeito: 2020 teve pandemia, 2021
recuperação, 2022 juros subindo. O rótulo depende da volatilidade realizada, então a proporção
de compra/venda/mantém **muda com o regime de mercado**. Um modelo treinado até 2019 vai
encontrar, em 2020, uma distribuição que ele nunca viu.

> 🎯 Isso tem nome: ***concept drift***. Em visão computacional, um gato continua sendo um gato.
> Em mercado, a relação entre entrada e saída **muda ao longo do tempo** — e é por isso que o
> walk-forward da Fase 5 não é preciosismo metodológico, é a única forma honesta de medir.

---

---

# FASE 1 — POR QUE Conv1D + LSTM

## §1.1 — As duas camadas fazem trabalhos diferentes

Nossa entrada é uma matriz $(T, F)$ = 30 dias × 23 features. Duas leituras possíveis:

| Camada | O que ela procura | Analogia |
|---|---|---|
| **Conv1D** | padrões **locais** no tempo: uma sequência de 3 dias em que a volatilidade sobe enquanto o volume cai | o "candle" da análise técnica, mas aprendido em vez de decorado |
| **LSTM** | **ordem e memória** de prazo mais longo: o que aconteceu no começo da janela muda o significado do que veio no fim | ler uma frase e lembrar do sujeito ao chegar no verbo |

Empilhar as duas é o padrão consagrado para série temporal: a convolução **comprime** 30 passos
ruidosos em poucos passos de resumo, e a LSTM percorre esse resumo. A LSTM sozinha sobre 30
passos de dados diários ruidosos tende a decorar; sobre 7 passos já filtrados, ela tem chance.

### 📐 A aritmética do encolhimento

$$\text{saída} = \left\lfloor\frac{\text{entrada} + 2p - k}{s}\right\rfloor + 1$$

Com `kernel=3, padding=1, stride=1`, a convolução **preserva** o comprimento. Cada `MaxPool1d(2)`
corta pela metade:

$$30 \xrightarrow{\text{conv}} 30 \xrightarrow{\text{pool}} 15 \xrightarrow{\text{conv}} 15 \xrightarrow{\text{pool}} 7$$

São esses 7 passos que a LSTM vai ler.

---
## §1.2 — A armadilha do formato: `(N, T, F)` × `(N, F, T)`

Este é **o** erro que aparece em toda primeira implementação de Conv1D sobre série temporal:

- Nosso tensor é `(N, T, F)` — amostra, tempo, feature. É o formato que a LSTM espera com
  `batch_first=True`.
- `nn.Conv1d` espera `(N, C, L)` — amostra, **canal**, comprimento. Ou seja: `(N, F, T)`.

Passar `(N, T, F)` direto para a `Conv1d` **não dá erro**: ela vai tratar os 30 dias como 30
canais e as 23 features como o eixo temporal. O código roda, a loss cai, e você está convolvendo
*features vizinhas na lista de colunas* — `rsi` com `pct_b` porque um veio depois do outro no
`build_features`. Um modelo treinado assim aprende a ordem alfabética das suas colunas.

> ⚠️ Por isso o `forward` faz `x.transpose(1, 2)` na entrada da convolução e desfaz antes da
> LSTM. Cada `transpose` está comentado no código: quando uma linha existe para satisfazer a
> convenção de uma biblioteca, o comentário tem que dizer **qual** convenção.

### 🤔 "Convolução centrada não vaza o futuro?"

Pergunta certa, e a resposta depende do que se prevê. Com `padding=1` a convolução é **centrada**:
a saída na posição $k$ mistura $k-1$, $k$ e $k+1$. Se o objetivo fosse produzir uma previsão
**para cada passo** da série, isso vazaria — a previsão de $k$ usaria $k+1$.

Aqui **não é o caso**: a janela inteira é passado em relação ao dia $t$ da decisão, e produzimos
**um** rótulo para a janela toda. Misturar dia 12 com dia 13 *dentro* da janela é combinar dois
dados que você já tinha. O que não pode entrar é informação posterior a $t$ — e isso foi
garantido lá atrás, no `truncation_test` e no split temporal.

> 💡 Se algum dia você mudar o alvo para "um rótulo por passo" (previsão *seq2seq*), volte aqui e
> troque por convolução **causal** (`padding` só à esquerda). A arquitetura certa depende do
> alvo, não do gosto.

---

---

# FASE 2 — O MODELO EM PYTORCH

In [ ]:
class CNNLSTM(nn.Module):
    """Conv1D (extrator local) -> LSTM (memória) -> Linear (logits de 3 classes).

    Formatos, passo a passo, com lote N:

        entrada             (N, 30, 23)   (T, F) -- como vem do .npz
        transpose(1,2)      (N, 23, 30)   (C, L) -- o que a Conv1d exige
        bloco conv 1        (N, 16, 15)
        bloco conv 2        (N, 32,  7)
        transpose(1,2)      (N,  7, 32)   (L, C) -- o que a LSTM exige (batch_first)
        LSTM (último passo) (N, 32)
        Linear              (N,  3)       logits
    """

    def __init__(self, n_features: int, n_classes: int = 3, canais=(16, 32),
                 kernel: int = 3, hidden: int = 32, camadas_lstm: int = 1,
                 dropout: float = 0.3):
        super().__init__()

        blocos, c_ant = [], n_features
        for c in canais:
            blocos += [
                # bias=False porque o BatchNorm em seguida já tem deslocamento próprio
                nn.Conv1d(c_ant, c, kernel_size=kernel, padding=kernel // 2, bias=False),
                nn.BatchNorm1d(c),
                nn.ReLU(inplace=True),
                nn.MaxPool1d(2),
            ]
            c_ant = c
        self.extrator = nn.Sequential(*blocos)

        self.lstm = nn.LSTM(c_ant, hidden, num_layers=camadas_lstm, batch_first=True)
        self.cabeca = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden, n_classes))

        self.hiperparametros = dict(n_features=n_features, n_classes=n_classes,
                                    canais=tuple(canais), kernel=kernel, hidden=hidden,
                                    camadas_lstm=camadas_lstm, dropout=dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Devolve LOGITS, sem softmax.

        `nn.CrossEntropyLoss` já aplica log-softmax internamente. Colocar softmax
        aqui aplicaria duas vezes: o treino fica lento e instável. O softmax só
        entra na hora de ler a saída como probabilidade (Fase 6).
        """
        z = x.transpose(1, 2)          # (N,T,F) -> (N,F,T): Conv1d quer canal no meio
        z = self.extrator(z)           # (N,C,T')
        z = z.transpose(1, 2)          # (N,C,T') -> (N,T',C): LSTM quer tempo no meio
        saida, _ = self.lstm(z)        # (N,T',H)
        return self.cabeca(saida[:, -1])   # só o último passo resume a janela

In [ ]:
# Rastreio de formatos ANTES de gastar tempo treinando -- com um tensor falso.
modelo_teste = CNNLSTM(n_features=len(FEATURES))
falso = torch.randn(4, T, len(FEATURES))

print("entrada         ", tuple(falso.shape))
z = falso.transpose(1, 2)
print("transpose(1,2)  ", tuple(z.shape), " <- (N, C, L) para a Conv1d")
for i in range(0, len(modelo_teste.extrator), 4):
    bloco = modelo_teste.extrator[i:i + 4]
    with torch.no_grad():
        z = bloco(z)
    print(f"bloco conv {i // 4 + 1}    ", tuple(z.shape))
print("transpose(1,2)  ", tuple(z.transpose(1, 2).shape), " <- (N, L, C) para a LSTM")
with torch.no_grad():
    logits = modelo_teste(falso)
print("saída (logits)  ", tuple(logits.shape))

n_par = sum(p.numel() for p in modelo_teste.parameters() if p.requires_grad)
print(f"\nparâmetros treináveis: {n_par:,}")

In [ ]:
# A comparação que decide o tamanho do modelo
print(f"{'ano':>6} {'n_treino':>9} {'parâmetros':>11} {'par/amostra':>12}")
for ano, f in sorted(folds.items()):
    n = len(f["y_train"])
    print(f"{ano:>6} {n:>9} {n_par:>11,} {n_par / n:>12.1f}")

print("\nEm visão computacional é comum ter 10x mais amostras que parâmetros.")
print("Aqui a razão está invertida por uma ordem de grandeza -- e não há como")
print("consertar isso com arquitetura: 10 anos de bolsa TÊM ~2.500 pregões.")

### 📌 A restrição que define o projeto inteiro

Dez anos de história diária de um papel são **cerca de 2.500 linhas**. Não existe truque de
arquitetura que resolva isso: você não pode baixar mais 2016. As saídas reais são três, e
nenhuma é gratuita:

| Saída | O que custa |
|---|---|
| **Modelo pequeno** (é o que fazemos) | capacidade limitada; padrões complexos ficam fora de alcance |
| **Vários papéis no mesmo modelo** | supõe que o padrão é o mesmo em PETR4 e WEGE3 — supõe demais |
| **Barra intradiária** (15min, 1h) | 30x mais dados, mas ruído muito maior e custo de execução pior |

Escolhemos a primeira, e é por isso que o modelo tem `canais=(16,32)` e `hidden=32` em vez dos
valores confortáveis de um tutorial de imagem. Toda regularização que vem a seguir — dropout,
*weight decay*, early stopping — existe para comprar um pouco de folga contra essa razão
desfavorável.

> 🎯 A pergunta "quantas amostras eu tenho por parâmetro?" deveria vir **antes** de escolher a
> arquitetura. Ela é a diferença entre um modelo que generaliza e um que decora o treino com
> perfeição e erra tudo no ano seguinte.

---

---

# FASE 3 — TREINANDO UM FOLD

Vamos treinar o fold mais recente e olhar o processo de perto. Três decisões que não são
convenções vazias:

**1. Pesos de classe na *loss*.** A classe `mantém` domina. Sem peso, o caminho de menor
resistência é prever `mantém` sempre — 45% de acerto e F1 macro péssimo. Os pesos vêm
calculados **só do treino** (Fase 4 do notebook anterior).

**2. Early stopping por **F1 macro** de validação**, não por acurácia nem por *loss*. Acurácia
premia quem chuta a classe majoritária; a *loss* de validação e o F1 nem sempre andam juntas
quando há peso de classe. Se a métrica de decisão é F1 macro, **pare** por F1 macro.

**3. Restaurar o melhor estado.** Ao parar, os pesos na memória são os da **última** época, que
já pioraram. Sem `load_state_dict` do melhor ponto, o early stopping não serve para nada —
é um erro silencioso e comum.

In [ ]:
def tensores_para_loader(X, y, batch=32, shuffle=False):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                       torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=batch, shuffle=shuffle)


def avaliar(modelo, loader) -> tuple:
    """Devolve (y_verdadeiro, y_previsto, probabilidades)."""
    modelo.eval()
    ys, preds, probs = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = modelo(xb.to(device))
            p = torch.softmax(logits, dim=1)        # softmax só AQUI, para ler probabilidade
            probs.append(p.cpu().numpy())
            preds.append(p.argmax(1).cpu().numpy())
            ys.append(yb.numpy())
    return np.concatenate(ys), np.concatenate(preds), np.concatenate(probs)


def treinar_fold(fold, epocas=80, paciencia=12, lr=1e-3, weight_decay=1e-4,
                 batch=32, semente=SEED, verboso=False):
    """Treina um modelo no fold e devolve (modelo, historico)."""
    torch.manual_seed(semente)                       # mesma inicialização a cada execução
    modelo = CNNLSTM(n_features=fold["X_train"].shape[2]).to(device)

    pesos = torch.tensor(fold["class_weights"], dtype=torch.float32, device=device)
    criterio = nn.CrossEntropyLoss(weight=pesos)
    otimizador = torch.optim.Adam(modelo.parameters(), lr=lr, weight_decay=weight_decay)

    tr = tensores_para_loader(fold["X_train"], fold["y_train"], batch, shuffle=True)
    va = tensores_para_loader(fold["X_val"], fold["y_val"], batch)

    melhor_f1, melhor_estado, espera = -1.0, None, 0
    hist = {"epoca": [], "loss_treino": [], "loss_val": [], "f1_val": []}

    for epoca in range(1, epocas + 1):
        modelo.train()
        soma = 0.0
        for xb, yb in tr:
            otimizador.zero_grad()
            perda = criterio(modelo(xb.to(device)), yb.to(device))
            perda.backward()
            nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)   # LSTM: evita gradiente explodir
            otimizador.step()
            soma += perda.item() * len(yb)

        y_va, p_va, _ = avaliar(modelo, va)
        f1_va = f1_score(y_va, p_va, average="macro")
        with torch.no_grad():
            loss_va = criterio(modelo(torch.tensor(fold["X_val"], dtype=torch.float32).to(device)),
                               torch.tensor(fold["y_val"], dtype=torch.long).to(device)).item()

        hist["epoca"].append(epoca)
        hist["loss_treino"].append(soma / len(fold["y_train"]))
        hist["loss_val"].append(loss_va)
        hist["f1_val"].append(f1_va)

        if f1_va > melhor_f1:
            melhor_f1, espera = f1_va, 0
            # copia para CPU: guardar referência ao state_dict vivo nao congela nada
            melhor_estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
            melhor_epoca = epoca
        else:
            espera += 1
            if espera >= paciencia:
                if verboso:
                    print(f"  early stopping na época {epoca}")
                break

    modelo.load_state_dict(melhor_estado)            # sem isto, o early stopping é decorativo
    hist["melhor_epoca"], hist["melhor_f1_val"] = melhor_epoca, melhor_f1
    return modelo, hist


ano_demo = max(folds)
t0 = time.time()
modelo_demo, hist_demo = treinar_fold(folds[ano_demo], verboso=True)
print(f"fold {ano_demo}: {len(hist_demo['epoca'])} épocas em {time.time() - t0:.1f}s")
print(f"melhor época = {hist_demo['melhor_epoca']} | F1 macro (validação) = {hist_demo['melhor_f1_val']:.3f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(hist_demo["epoca"], hist_demo["loss_treino"], label="treino")
ax1.plot(hist_demo["epoca"], hist_demo["loss_val"], label="validação")
ax1.axvline(hist_demo["melhor_epoca"], color="gray", ls="--", lw=1, label="melhor época")
ax1.set(title=f"Loss — fold {ano_demo}", xlabel="época", ylabel="cross-entropy")
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(hist_demo["epoca"], hist_demo["f1_val"], color="tab:green")
ax2.axvline(hist_demo["melhor_epoca"], color="gray", ls="--", lw=1)
ax2.axhline(1 / 3, color="tab:red", ls=":", lw=1, label="chute uniforme (~0,33)")
ax2.set(title="F1 macro na validação", xlabel="época", ylabel="F1 macro")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()

### 🔎 Como ler essas curvas (e o que elas dizem aqui)

**No gráfico da esquerda**, o padrão de livro é: as duas loss caem juntas, depois a de validação
para de cair e sobe enquanto a de treino continua descendo. Esse descolamento é
**sobreajuste** — o modelo passou a decorar. A linha tracejada marca onde o early stopping
congelou os pesos, e ela costuma ficar **antes** do fim do gráfico: as épocas à direita foram
treinadas e descartadas, que é exatamente o serviço da paciência.

**No gráfico da direita**, o F1 macro de validação. A linha pontilhada em 0,33 é o chute
uniforme entre três classes. Repare em duas coisas:

- a curva é **ruidosa**, oscilando de época para época. Com ~200 janelas de validação, cada
  ponto tem incerteza grande. Escolher "a melhor época" nessa curva é, em parte, escolher o
  melhor ruído — e é por isso que o resultado de validação **não é** a medida final de qualidade;
- o patamar fica **perto** da linha de referência. Não estamos diante de um modelo que
  descobriu o mercado.

> ⚠️ Um detalhe metodológico que passa batido: o early stopping **usa** a validação para
> escolher a época. Isso torna a métrica de validação **otimista** — ela foi maximizada, não
> observada. É por isso que existe um terceiro conjunto, o teste, em que não se toca. Se você
> ajustar o modelo olhando o teste, ele deixa de ser teste e vira validação com outro nome.

---

---

# FASE 4 — AVALIAR CONTRA BASELINES QUE NÃO SÃO FÁCEIS

Um F1 macro de 0,35 é bom ou ruim? A pergunta não tem resposta sozinha. Precisa de referência —
e a referência tem que ser **honesta**, não uma que você escolheu para ganhar.

Quatro candidatos:

| Baseline | Por que ele importa |
|---|---|
| **classe majoritária do treino** | o que um modelo preguiçoso acerta sem aprender nada |
| **sorteio com as proporções do treino** | o acaso *informado* pela distribuição de classes |
| **sempre `compra`** | o equivalente a comprar e segurar, na linguagem da classificação |
| **regressão logística** (notebook anterior) | um modelo linear sobre a última linha da janela |

O terceiro e o quarto parecem bobos e não são: `sempre compra` tem acurácia igual à frequência da
classe `compra`, e a classe majoritária tem acurácia igual à frequência de `mantém` — em torno de
40% cada. Uma rede com 39% de acurácia **não está batendo** nenhum dos dois.

E o **sorteio com prior** é o baseline mais honesto de todos em F1 macro: ele usa as três classes,
nas proporções certas, sem aprender nada. Bater esse é a barra mínima para dizer que houve
aprendizado.

In [ ]:
fold = folds[ano_demo]
loader_te = tensores_para_loader(fold["X_test"], fold["y_test"])
y_te, p_rede, prob_rede = avaliar(modelo_demo, loader_te)

rng = np.random.default_rng(SEED)
prior = np.bincount(fold["y_train"], minlength=3) / len(fold["y_train"])

referencias = {
    "rede CNN+LSTM": p_rede,
    "classe majoritária": np.full_like(y_te, int(np.argmax(prior))),
    "sorteio com prior": rng.choice(3, size=len(y_te), p=prior),
    "sempre compra": np.full_like(y_te, 2),
}

placar = pd.DataFrame([
    {"modelo": nome,
     "f1_macro": f1_score(y_te, pred, average="macro"),
     "acurácia": (pred == y_te).mean()}
    for nome, pred in referencias.items()
]).set_index("modelo").sort_values("f1_macro", ascending=False)

print(f"conjunto de TESTE do fold {ano_demo} ({len(y_te)} janelas)\n")
placar.round(3)

In [ ]:
print("matriz de confusão da rede (linhas = real, colunas = previsto)\n")
cm = confusion_matrix(y_te, p_rede, labels=[0, 1, 2])
print(pd.DataFrame(cm, index=[f"real {n}" for n in NOMES],
                   columns=[f"prev {n}" for n in NOMES]).to_string())

print()
print(classification_report(y_te, p_rede, labels=[0, 1, 2],
                            target_names=NOMES, zero_division=0))

### ⚠️ Antes de julgar: este é **um** fold, e provavelmente um ruim

Na execução de referência a rede aparece **atrás do sorteio com prior** neste fold, e a regra
"sempre compra" tem a melhor acurácia de todas. Se você parasse aqui, a conclusão seria "a rede
não aprendeu nada".

Não pare aqui. Escolhemos de propósito o fold **mais recente** (`ano_demo = max(folds)`), e ele
é, na execução de referência, o pior dos sete. Um fold é uma amostra de tamanho **um**: ele não
distingue um modelo ruim de um ano difícil. A Fase 5 mede os sete, e o retrato muda.

> 🎯 A tentação oposta é igualmente comum e pior: rodar vários folds, achar o melhor e mostrar
> aquele. Se um fold não serve para condenar, também não serve para absolver.

### 🔎 O que procurar nessa matriz

Três leituras, em ordem de importância:

1. **A rede usa as três classes?** Se uma coluna estiver quase vazia, o modelo desistiu daquela
   classe — o peso na *loss* não foi suficiente. Se uma coluna concentrar quase tudo, ele
   colapsou para uma resposta só, e o F1 macro vai denunciar mesmo com acurácia razoável.
2. **Os erros são simétricos?** Confundir `mantém` com `compra` custa uma operação desnecessária.
   Confundir **`venda` com `compra`** é o erro caro: você compra o que deveria evitar. Olhe
   especificamente as celas `real venda / prev compra` e `real compra / prev venda`.
3. **A diagonal se destaca?** Compare com o que o sorteio produziria. Uma diagonal só um pouco
   mais forte que o acaso é o resultado típico — e não é fracasso do aluno, é a natureza do
   problema.

> 🎯 **Precisão da classe `compra` é a métrica que vira dinheiro.** O F1 macro trata as três
> classes como igualmente importantes, mas a estratégia da Fase 6 só age quando o modelo diz
> `compra`. Se a precisão dessa classe estiver em torno de 1/3, cada operação é uma aposta
> ligeiramente pior que cara-ou-coroa — e ainda vai pagar custo.

---

---

# FASE 5 — WALK-FORWARD: SETE MODELOS, SETE ANOS

Um fold é uma amostra de tamanho um. O modelo pode ter ido bem em 2026 por sorte, ou mal por
azar. O walk-forward treina um modelo **por ano de teste**, sempre com dados **anteriores** —
é a única forma de obter sete medidas realmente fora da amostra.

Note o que **não** estamos fazendo: escolher a arquitetura olhando o resultado dos sete anos e
depois apresentar esse resultado como "fora da amostra". Isso é o mesmo vazamento de antes, só
mais lento. Os hiperparâmetros da Fase 2 foram fixados por critério de tamanho de amostra,
antes de ver qualquer teste.

In [ ]:
t0 = time.time()
resultados, previsoes = [], []
rng_wf = np.random.default_rng(SEED)

for ano in sorted(folds):
    f = folds[ano]
    modelo, hist = treinar_fold(f, semente=SEED)          # mesma semente: comparável entre anos
    y_true, y_pred, prob = avaliar(modelo, tensores_para_loader(f["X_test"], f["y_test"]))

    prior_ano = np.bincount(f["y_train"], minlength=3) / len(f["y_train"])
    maj_ano = int(np.argmax(prior_ano))
    sorteio = rng_wf.choice(3, size=len(y_true), p=prior_ano)
    resultados.append({
        "ano": ano,
        "épocas": len(hist["epoca"]),
        "f1_val": hist["melhor_f1_val"],
        "f1_teste": f1_score(y_true, y_pred, average="macro"),
        "f1_sorteio": f1_score(y_true, sorteio, average="macro"),
        "f1_majoritária": f1_score(y_true, np.full_like(y_true, maj_ano), average="macro"),
        "f1_sempre_compra": f1_score(y_true, np.full_like(y_true, 2), average="macro"),
        "acurácia": (y_true == y_pred).mean(),
        "acc_majoritária": (y_true == maj_ano).mean(),
        "%_prev_compra": (y_pred == 2).mean(),
    })
    previsoes.append(pd.DataFrame({
        "data": pd.to_datetime(f["dates_test"]),
        "ano_fold": ano, "y_true": y_true, "y_pred": y_pred,
        "prob_venda": prob[:, 0], "prob_mantem": prob[:, 1], "prob_compra": prob[:, 2],
    }))
    print(f"  {ano}: {len(hist['epoca']):3d} épocas | F1 teste {resultados[-1]['f1_teste']:.3f}")

wf = pd.DataFrame(resultados).set_index("ano")
oos = pd.concat(previsoes, ignore_index=True).sort_values("data").set_index("data")

print(f"\ntreino completo em {time.time() - t0:.0f}s")
print(f"previsões fora da amostra: {len(oos)} dias, de {oos.index.min().date()} a {oos.index.max().date()}")
wf.round(3)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
x = np.arange(len(wf))

ax1.bar(x - 0.26, wf["f1_teste"], width=0.26, label="rede CNN+LSTM")
ax1.bar(x, wf["f1_sorteio"], width=0.26, label="sorteio com prior")
ax1.bar(x + 0.26, wf["f1_majoritária"], width=0.26, label="classe majoritária")
ax1.set_xticks(x, wf.index)
ax1.set(title="F1 macro fora da amostra", xlabel="ano de teste", ylabel="F1 macro")
ax1.legend(fontsize=9); ax1.grid(alpha=0.3, axis="y")

ax2.bar(x - 0.16, wf["acurácia"], width=0.32, label="rede CNN+LSTM")
ax2.bar(x + 0.16, wf["acc_majoritária"], width=0.32, label="classe majoritária")
ax2.set_xticks(x, wf.index)
ax2.set(title="Acurácia fora da amostra", xlabel="ano de teste", ylabel="acurácia")
ax2.legend(fontsize=9); ax2.grid(alpha=0.3, axis="y")

plt.tight_layout(); plt.show()

print("médias em 7 anos fora da amostra:")
print(wf[["f1_teste", "f1_sorteio", "f1_majoritária", "f1_sempre_compra",
          "acurácia", "acc_majoritária"]].mean().round(3).to_string())
print()
for rival, metrica in (("f1_sorteio", "f1_teste"), ("f1_majoritária", "f1_teste"),
                       ("acc_majoritária", "acurácia")):
    venceu = int((wf[metrica] > wf[rival]).sum())
    print(f"anos em que a rede venceu {rival:17s} ({metrica}): {venceu} de {len(wf)}")

### 📌 Duas métricas, duas conclusões opostas — e as duas são verdadeiras

Médias dos sete anos na execução de referência (dados até 21/09/2026):

| Métrica | Rede | Sorteio com prior | Classe majoritária |
|---|---|---|---|
| **F1 macro** | 0,35 | 0,31 | 0,20 |
| **Acurácia** | 0,38 | — | **0,42** |

Leia devagar, porque as duas linhas discordam:

- **Em F1 macro a rede ganha** de todos os baselines. Mas cuidado com a comparação contra a classe
  majoritária: um preditor que só responde uma classe tem F1 zero nas outras duas, então seu macro
  fica preso perto de 0,2 **por construção**. Ganhar dele é quase automático. A comparação que vale
  é contra o **sorteio com prior** — e ali a margem é de 0,04, vencendo em **4 dos 7 anos**. Quatro
  de sete é praticamente cara-ou-coroa.
- **Em acurácia a rede perde** para "responda sempre `mantém`", em 5 dos 7 anos. Ou seja: ela não
  acerta mais vezes que a regra trivial — ela **distribui melhor os acertos entre as três
  classes**.

> 🎯 Isso não é contradição, é o significado de cada métrica. F1 macro pergunta "você é
> competente nas três classes?"; acurácia pergunta "você acerta mais vezes?". Um modelo pode
> melhorar uma e piorar a outra, e **qual delas importa depende do uso**. No nosso caso o uso é
> operar quando o modelo diz `compra` — então nem uma nem outra é a métrica final. A métrica
> final é dinheiro, e ela vem na Fase 6.

### Olhe a dispersão antes da média

Os anos variam muito — e o pior deles costuma ser o mais recente, o mesmo fold que usamos na
Fase 4. Aquela avaliação isolada, em que a rede ficou **atrás do sorteio**, não estava errada:
era um ano ruim, visto de perto. É exatamente por isso que uma medida em um fold não decide nada.

> ⚠️ E se a tentação agora for mexer na arquitetura até o `f1_teste` melhorar, pare. Você tem 7
> números; ajustar o modelo até eles ficarem bons é ajustar **aos 7 anos de teste**, e o próximo
> ano real não vai colaborar. O lugar de experimentar é a validação — e o preço de cada
> experimento é um pouco mais de otimismo na estimativa final.

Repare também em `f1_val` contra `f1_teste`: a validação é quase sempre melhor. Já sabíamos por
quê — o early stopping **escolheu** a época de melhor validação. Ver a diferença medida, ano a
ano, é diferente de saber a teoria.

---

---

# FASE 6 — DO ACERTO AO DINHEIRO: BACKTEST COM CUSTOS

## §6.1 — As regras do jogo, declaradas antes de rodar

Um backtest é uma **simulação**, e toda simulação vale o que valem suas premissas. As nossas,
explícitas:

| Premissa | Escolha | Por quê |
|---|---|---|
| **Posições** | comprado ou fora (*long/flat*) | venda a descoberto na B3 tem aluguel e risco de recompra; fica como exercício |
| **Gatilho** | entra quando a previsão é `compra`, sai caso contrário | a decisão é reavaliada todo dia |
| **Execução** | **abertura do dia seguinte** | a decisão usa o fechamento de $t$; você não consegue negociar naquele fechamento que acabou de observar |
| **Preço** | série **ajustada** (o mesmo fator `Adj Close / Close` do notebook de dividendos) | proventos precisam entrar no retorno |
| **Emolumentos** | 0,032% por operação | taxa da B3 sobre o valor financeiro |
| **Corretagem** | R$ 0 | a maioria das corretoras zerou para ações |
| **Slippage** | 0,05% por operação | você não executa no preço que viu; em papel líquido essa é uma estimativa branda |
| **IR** | 15% sobre ganho mensal, isento se as vendas do mês ≤ R$ 20 mil | regra de *swing trade* para ações (pessoa física) |
| **Capital** | R$ 10.000, sem aportes | tamanho que não move o preço |
| **Papel** | o que vier gravado nos tensores | escolher aqui permitiria descasar sinal e preço |

> ⚠️ **Sobre o momento da execução.** É aqui que a maioria dos backtests amadores mente, e sem
> perceber: usa-se o fechamento de $t$ para decidir e o **mesmo** fechamento para comprar. São
> os mesmos dados, então não parece vazamento — mas na prática você teria que enviar a ordem
> antes de saber o fechamento. Deslocar a execução para a abertura seguinte custa retorno no
> papel e é o que aconteceria na sua conta. Medir o tamanho dessa diferença é um dos exercícios.

In [ ]:
def carregar_precos(ticker="PETR4.SA", caminho=Path("data/raw/b3_cotacoes.parquet")):
    """OHLC AJUSTADO do papel, reconstruído a partir do parquet do primeiro notebook.

    O parquet guarda Close e Adj Close separados (auto_adjust=False). O fator
    f = Adj Close / Close é o mesmo do notebook de dividendos: aplicá-lo a
    open/high/low reproduz exatamente o que `auto_adjust=True` devolve -- que é a
    série sobre a qual features e rótulos foram construídos. Misturar preço ajustado
    com preço de tela daria um retorno que ninguém realizou.
    """
    if caminho.exists():
        bruto = pd.read_parquet(caminho)
        bruto = bruto[bruto["Ticker"] == ticker].sort_values("Date").set_index("Date")
        f = (bruto["Adj Close"] / bruto["Close"]).values
        px = pd.DataFrame({
            "open": bruto["Open"].values * f,
            "high": bruto["High"].values * f,
            "low": bruto["Low"].values * f,
            "close": bruto["Adj Close"].values,
        }, index=bruto.index)
        return px, f"parquet local ({caminho})"

    import yfinance as yf                              # plano B, se não rodou o notebook 1
    df = yf.download(ticker, start="2016-01-01", auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.rename(columns=str.lower)[["open", "high", "low", "close"]]
    return df.dropna(), "yfinance (download ao vivo)"


# O papel NAO e escolhido aqui: ele vem dos tensores (Fase 0). Escrever o ticker a
# mao neste notebook abriria a porta para o pior erro possivel -- sinal de um ativo
# aplicado ao preco de outro, sem erro nenhum na tela.
TICKER_DECLARADO = "PETR4.SA"        # só serve para tensores antigos, sem procedência
TICKER = PROC["ticker"] or TICKER_DECLARADO

if PROC["ticker"] is None:
    print(f"[aviso] tensores sem procedência: usando o ticker declarado ({TICKER}).")
    print("        Confirme que é o mesmo do notebook de preparação, ou rode-o de novo.\n")

precos, fonte = carregar_precos(TICKER)
print(f"{TICKER} | fonte dos preços: {fonte}")
print(f"{len(precos)} pregões | {precos.index.min().date()} → {precos.index.max().date()}")

# Todo dia com sinal precisa ter preço. Dois artefatos, duas datas de coleta: os
# tensores vêm do yfinance no dia em que você rodou a preparação; o parquet vem da
# coleta, que pode ser mais antiga. A diferença aparece como sinal sem preço.
faltantes = oos.index.difference(precos.index)

if len(faltantes) == 0:
    print(f"\ntodas as {len(oos)} datas de sinal têm preço correspondente")
else:
    fracao = len(faltantes) / len(oos)
    apenas_cauda = bool((faltantes > precos.index.max()).all())
    if fracao > 0.05 or not apenas_cauda:
        # Buracos no meio, ou muitos: isto não é defasagem, é papel/arquivo errado.
        raise ValueError(
            f"{len(faltantes)} de {len(oos)} datas de sinal ({fracao:.1%}) não têm preço, "
            f"e não estão só no fim da série. Sinal é de {TICKER} — o parquet é do mesmo papel? "
            f"Exemplos: {faltantes[:5].tolist()}")
    print(f"\n[aviso] o parquet de preços termina em {precos.index.max().date()}, "
          f"antes das últimas {len(faltantes)} datas de sinal.")
    print("        O backtest vai até onde há preço. Para cobrir tudo, rode "
          "coleta_dados_b3.ipynb de novo.")
    oos = oos.loc[oos.index.isin(precos.index)]
    print(f"        sinal truncado: {len(oos)} dias, até {oos.index.max().date()}")

In [ ]:
EMOL = 0.00032          # emolumentos B3 por operação
SLIPPAGE = 0.0005       # escorregamento estimado por operação
IR_PCT = 0.15           # swing trade, ações, pessoa física
ISENCAO_MES = 20_000.0  # vendas mensais até este valor são isentas
CAPITAL = 10_000.0


def backtest(precos: pd.DataFrame, sinal: pd.Series, capital=CAPITAL,
             emol=EMOL, slippage=SLIPPAGE, ir_pct=IR_PCT, isencao=ISENCAO_MES,
             classe_compra=2) -> dict:
    """Long/flat, decisão no fechamento de t, execução na ABERTURA de t+1.

    Devolve patrimônio diário, operações, custos e IM apurado por mês.
    """
    datas = precos.loc[sinal.index.min():].index        # começa no 1o dia com sinal
    custo_lado = emol + slippage

    caixa, cotas, preco_medio = capital, 0.0, 0.0
    patrimonio, operacoes = [], []
    vendas_mes, ganho_mes = {}, {}

    for i in range(len(datas) - 1):                     # -1: precisa existir t+1
        dia, exec_dia = datas[i], datas[i + 1]
        preco_exec = precos.at[exec_dia, "open"]

        # Sem sinal naquele dia (ex.: aquecimento de janela no início de cada fold)
        # => fica FORA. Não inventamos posição onde o modelo não opinou.
        quer_comprado = int(sinal.get(dia, 1) == classe_compra)
        esta_comprado = int(cotas > 0)

        if quer_comprado != esta_comprado:
            if quer_comprado:                           # ---- COMPRA
                valor = caixa
                custo = valor * custo_lado
                cotas = (valor - custo) / preco_exec
                preco_medio = preco_exec * (1 + custo_lado)   # custo embutido no preço médio
                caixa = 0.0
                operacoes.append({"data": exec_dia, "tipo": "compra", "preço": preco_exec,
                                  "valor": valor, "custo": custo, "resultado": np.nan})
            else:                                       # ---- VENDA
                valor = cotas * preco_exec
                custo = valor * custo_lado
                resultado = (valor - custo) - cotas * preco_medio
                caixa = valor - custo
                mes = exec_dia.to_period("M")
                vendas_mes[mes] = vendas_mes.get(mes, 0.0) + valor
                ganho_mes[mes] = ganho_mes.get(mes, 0.0) + resultado
                cotas = 0.0
                operacoes.append({"data": exec_dia, "tipo": "venda", "preço": preco_exec,
                                  "valor": valor, "custo": custo, "resultado": resultado})

        patrimonio.append({"data": exec_dia,
                           "patrimonio": caixa + cotas * precos.at[exec_dia, "close"],
                           "comprado": int(cotas > 0)})

    eq = pd.DataFrame(patrimonio).set_index("data")
    ops = pd.DataFrame(operacoes)

    # IR mês a mês: a isenção vale sobre o VOLUME DE VENDAS do mês, não sobre o lucro
    ir_total, ir_detalhe = 0.0, []
    for mes in sorted(set(vendas_mes) | set(ganho_mes)):
        vendas, ganho = vendas_mes.get(mes, 0.0), ganho_mes.get(mes, 0.0)
        isento = vendas <= isencao
        devido = 0.0 if (isento or ganho <= 0) else ganho * ir_pct
        ir_total += devido
        ir_detalhe.append({"mês": str(mes), "vendas": vendas, "ganho": ganho,
                           "isento": isento, "ir": devido})

    return {"equity": eq, "operacoes": ops, "ir_total": ir_total,
            "ir_detalhe": pd.DataFrame(ir_detalhe),
            "custos": float(ops["custo"].sum()) if len(ops) else 0.0,
            "patrimonio_final_sem_ir": float(eq["patrimonio"].iloc[-1]),
            "patrimonio_final": float(eq["patrimonio"].iloc[-1]) - ir_total}


sinal_rede = oos["y_pred"]
sinal_comprado = pd.Series(2, index=oos.index)          # sempre comprado = buy & hold

bt_rede = backtest(precos, sinal_rede)
bt_bruto = backtest(precos, sinal_rede, emol=0.0, slippage=0.0, ir_pct=0.0)
bt_bh = backtest(precos, sinal_comprado)

print(f"operações da rede : {len(bt_rede['operacoes'])}")
print(f"operações do B&H  : {len(bt_bh['operacoes'])}")
print(f"custos da rede    : R$ {bt_rede['custos']:,.2f}")
print(f"IR da rede        : R$ {bt_rede['ir_total']:,.2f}")

In [ ]:
def metricas(bt, nome, dias_ano=252) -> dict:
    eq = bt["equity"]["patrimonio"]
    ret = eq.pct_change().dropna()
    anos = len(eq) / dias_ano
    total = eq.iloc[-1] / eq.iloc[0] - 1
    # Sharpe com risk-free = 0. No Brasil isso favorece a estratégia: o CDI não é zero.
    sharpe = (ret.mean() / ret.std() * np.sqrt(dias_ano)) if ret.std() > 0 else np.nan
    dd = (eq / eq.cummax() - 1).min()
    ops = bt["operacoes"]
    vendas = ops[ops["tipo"] == "venda"] if len(ops) else ops

    liquido = bt["patrimonio_final"] / eq.iloc[0] - 1
    return {
        "estratégia": nome,
        "retorno_%": total * 100,
        "retorno_líq_IR_%": liquido * 100,
        "CAGR_%": ((1 + total) ** (1 / anos) - 1) * 100,
        "vol_a.a._%": ret.std() * np.sqrt(dias_ano) * 100,
        "sharpe": sharpe,
        "max_DD_%": dd * 100,
        "exposição_%": bt["equity"]["comprado"].mean() * 100,
        "n_operações": len(ops),
        "acerto_por_trade_%": (vendas["resultado"] > 0).mean() * 100 if len(vendas) else np.nan,
        "custos_R$": bt["custos"],
        "IR_R$": bt["ir_total"],
    }


painel = pd.DataFrame([
    metricas(bt_bruto, "rede — SEM custos"),
    metricas(bt_rede, "rede — com custos"),
    metricas(bt_bh, "buy & hold"),
]).set_index("estratégia")

print(f"período: {bt_rede['equity'].index.min().date()} a {bt_rede['equity'].index.max().date()}"
      f"  ({len(bt_rede['equity']) / 252:.1f} anos)\n")
painel.round(2)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                               gridspec_kw={"height_ratios": [2, 1]})

for bt, nome, estilo in ((bt_bruto, "rede — sem custos", "--"),
                         (bt_rede, "rede — com custos", "-"),
                         (bt_bh, "buy & hold", "-")):
    eq = bt["equity"]["patrimonio"]
    ax1.plot(eq.index, eq, ls=estilo, lw=1.4, label=nome)

ax1.axhline(CAPITAL, color="gray", ls=":", lw=1)
ax1.set(title=f"Patrimônio — {TICKER}, fora da amostra", ylabel="R$")
ax1.legend(); ax1.grid(alpha=0.3)

eq_r = bt_rede["equity"]["patrimonio"]
eq_b = bt_bh["equity"]["patrimonio"]
ax2.fill_between(eq_r.index, (eq_r / eq_r.cummax() - 1) * 100, 0, alpha=0.6,
                 label="rede (com custos)")
ax2.plot(eq_b.index, (eq_b / eq_b.cummax() - 1) * 100, color="tab:green", lw=1,
         label="buy & hold")
ax2.set(title="Drawdown", xlabel="data", ylabel="%")
ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# O CDI é o piso de comparação no Brasil: um Sharpe calculado com risk-free zero
# dá à estratégia um crédito que ela não tem.
caminho_macro = Path("data/raw/b3_macro.parquet")
if caminho_macro.exists():
    macro = pd.read_parquet(caminho_macro)
    selic = (macro[macro["Indicador"] == "selic"]
             .set_index("Date")["Valor"].sort_index())
    ini, fim = bt_rede["equity"].index.min(), bt_rede["equity"].index.max()
    janela = selic.loc[ini:fim]
    acumulado_cdi = float(np.prod(1 + janela.values / 100) - 1)

    print(f"Selic acumulada de {ini.date()} a {fim.date()}: {acumulado_cdi * 100:,.1f}%")
    print(f"  rede com custos e IR : {(bt_rede['patrimonio_final'] / CAPITAL - 1) * 100:,.1f}%")
    print(f"  buy & hold           : {(bt_bh['patrimonio_final'] / CAPITAL - 1) * 100:,.1f}%")
    print(f"\nR$ {CAPITAL:,.0f} parados na Selic teriam virado "
          f"R$ {CAPITAL * (1 + acumulado_cdi):,.0f} — sem risco, sem modelo e sem trabalho.")
else:
    print("data/raw/b3_macro.parquet não encontrado — rode coleta_dados_b3.ipynb")
    print("para comparar com a Selic. Sem esse piso, o Sharpe engana.")

### 📌 A conta que explica o resultado

Na execução de referência (PETR4, 2020-02 a 2026-09, capital de R$ 10.000):

| Estratégia | Retorno | Depois do IR | Exposição | Operações | Sharpe | Max DD |
|---|---|---|---|---|---|---|
| rede, **sem** custos | +87% | +87% | 37% | 264 | 0,51 | −49% |
| rede, **com** custos | +50% | **+32%** | 37% | 264 | 0,38 | −49% |
| buy & hold | +224% | **+205%** | 89% | 13 | 0,66 | −63% |
| Selic acumulada | — | **+89%** | — | 0 | — | 0% |

Três leituras, na ordem em que doem:

**1. A fricção levou 55 pontos percentuais.** De +87% para +32%: R$ 2.522 de custos e R$ 1.880 de
imposto, somando **R$ 4.402 — 44% do capital inicial**, pagos ao longo de 264 operações. Não é
detalhe de segunda ordem; é a maior linha do resultado.

**2. Ficar fora do mercado custou mais que os custos.** A estratégia esteve investida **37% dos
dias**; o buy & hold, 89%. Em um papel que subiu 224% no período, cada dia fora é um dia de alta
perdida. Note o consolo parcial: o *drawdown* máximo caiu de −63% para −49% e a volatilidade de
40% para 26% — a estratégia **é** menos arriscada. Só não o suficiente: o Sharpe (0,38) continua
abaixo do buy & hold (0,66).

**3. Perdeu para não fazer nada.** A Selic acumulou +89% no mesmo período, contra +32% da
estratégia. Este é o piso brasileiro: renda variável que não bate o CDI não perdeu para o buy &
hold — perdeu para **deixar o dinheiro parado**, com uma fração do risco. E é por isso que o
Sharpe que calculamos, com *risk-free* igual a zero, **superestima** toda estratégia brasileira.

### ⚠️ Os seus números vão ser outros — e o quanto eles mudam é a informação mais importante aqui

Rodei este pipeline duas vezes, **com o mesmo código e a mesma semente**, mudando apenas a data de
coleta dos dados: uma com histórico até 18/09/2026, outra até 21/09/2026. **Três dias de
diferença.** O que aconteceu:

| | até 18/09 | até 21/09 |
|---|---|---|
| retorno líquido | **+62%** | **+32%** |
| anos em que a rede bateu o sorteio | 6 de 7 | 4 de 7 |
| custos + IR | R$ 5.537 | R$ 4.402 |

O buy & hold foi **idêntico** nas duas (+205%), o que prova que os preços do backtest não mudaram.
A diferença inteira veio dos **tensores**: três dias a mais deslocam as fronteiras dos folds do
walk-forward, o que troca alguns dias de treino, o que muda os pesos aprendidos, o que muda 264
decisões, que se acumulam em 6,5 anos.

> 🎯 **Um resultado que muda por metade ao acrescentar três dias de dados não é um resultado — é
> ruído com aparência de conclusão.** Se você só tivesse rodado a primeira vez, teria anotado
> "+62%, vantagem em 6 de 7 anos" e acreditado. Rode duas vezes, com recortes ligeiramente
> diferentes, **antes** de acreditar em qualquer backtest — inclusive neste.
>
> O que sobreviveu às duas execuções: a rede perde para o buy & hold, perde para o CDI, e a
> fricção come metade do retorno bruto. **Essas** são as conclusões. O número exato não é.

> ⚠️ Repare também no artefato do `exposição_%`: no começo de cada ano de teste há ~30 pregões
> sem sinal, porque cada bloco do walk-forward precisa de $T$ = 30 dias para montar a primeira
> janela. Nesses dias a estratégia fica fora por **falta de dado**, não por decisão. Um sistema
> em produção carregaria a janela pela fronteira — e corrigir isso é um dos exercícios.

---
## §6.2 — O que a taxa de acerto tem a ver com o lucro (menos do que você imagina)

In [ ]:
vendas = bt_rede["operacoes"].query("tipo == 'venda'").copy()
if len(vendas):
    # resultado bruto por trade, em % do valor investido
    vendas["ret_%"] = vendas["resultado"] / (vendas["valor"] - vendas["resultado"]) * 100
    ganhos = vendas.loc[vendas["resultado"] > 0, "ret_%"]
    perdas = vendas.loc[vendas["resultado"] <= 0, "ret_%"]

    custo_ida_volta = 2 * (EMOL + SLIPPAGE) * 100
    print(f"operações fechadas       : {len(vendas)}")
    print(f"taxa de acerto           : {(vendas['resultado'] > 0).mean() * 100:.1f}%")
    print(f"ganho médio quando acerta: {ganhos.mean():+.2f}%" if len(ganhos) else "")
    print(f"perda média quando erra  : {perdas.mean():+.2f}%" if len(perdas) else "")
    print(f"resultado médio por trade: {vendas['ret_%'].mean():+.3f}%")
    print(f"\ncusto de ida e volta     : {custo_ida_volta:.3f}% por operação")
    print(f"dias médios por posição  : "
          f"{bt_rede['equity']['comprado'].sum() / max(len(vendas), 1):.1f}")

    # Qual taxa de acerto seria necessária para empatar, dados ganho e perda médios?
    if len(ganhos) and len(perdas):
        g, p = ganhos.mean(), abs(perdas.mean())
        necessaria = p / (g + p) * 100
        print(f"\ntaxa de acerto de equilíbrio (com esses ganho/perda médios): {necessaria:.1f}%")
        print(f"taxa de acerto observada                                  : "
              f"{(vendas['resultado'] > 0).mean() * 100:.1f}%")
else:
    print("nenhuma operação fechada no período")

### 🎯 O paradoxo: cada operação é boa, e o conjunto perde

Os números da execução de referência, por operação:

| Medida | Valor |
|---|---|
| operações fechadas | 132 |
| taxa de acerto | **60,6%** |
| ganho médio quando acerta | +3,01% |
| perda média quando erra | −3,31% |
| resultado médio por operação (já com custos) | **+0,52%** |
| custo de ida e volta | 0,164% |
| taxa de acerto de **equilíbrio** | 52,3% |

Leia o que isso diz: a taxa de acerto observada (60,6%) está **oito pontos acima** do ponto de
equilíbrio (52,3%), e o resultado médio por operação é **três vezes** o custo de ida e volta. A
vantagem é real e **sobrevive aos custos no nível da operação**.

Note também por que o ponto de equilíbrio não é 50%: as perdas médias (−3,31%) são **maiores** que
os ganhos médios (+3,01%). Com ganho e perda assimétricos, você precisa acertar mais vezes só para
empatar — e o *stop* implícito no rótulo (a barreira inferior) é o que produz essa assimetria.

E ainda assim o resultado final perdeu para o buy & hold e para a Selic. Como?

**1. Custo de oportunidade.** 132 operações de ~4,7 dias somam 37% do tempo investido. Os 63%
restantes o dinheiro ficou parado — enquanto o papel subia. Uma vantagem por operação não
compensa a ausência de operações.

**2. Efeito da variância (*variance drag*).** A média aritmética de +0,52% **não** se transforma
em $(1{,}0052)^{132}$. Com ganhos de +3,0% e perdas de −3,3% alternando, o crescimento composto é
bem menor que a média sugere: ganhar 3% e perder 3% não volta ao ponto de partida — volta a
0,999. Quanto maior a variância por operação, maior essa erosão.

**3. Assimetria do imposto.** A alíquota incide nos meses de lucro; os meses de prejuízo não
devolvem nada no nosso modelo simplificado (na vida real o prejuízo é compensável nos meses
seguintes, o que ameniza mas não elimina a assimetria). E note a isenção: ela depende do **volume
de vendas** do mês, não do lucro — girar muito estoura os R$ 20 mil e liga o imposto mesmo com
capital pequeno. Na referência, **22 dos 58 meses** com venda pagaram IR; os outros 29 ficaram
isentos justamente por terem girado menos.

> 📌 **A lição transferível:** "tenho vantagem estatística" e "ganho dinheiro" são afirmações
> **diferentes**, e a segunda não decorre da primeira. Entre elas estão o giro, o tempo fora, a
> variância e o imposto. O mesmo vale para qualquer sistema que toma muitas decisões pequenas —
> lances de anúncio, roteamento, precificação dinâmica: o custo unitário da ação define o tamanho
> mínimo da vantagem que vale a pena, e a frequência define se ela chega ao resultado.

---

---

# FASE 7 — O DIAGNÓSTICO HONESTO

## §7.1 — Onde este pipeline está, em uma tabela

| Etapa | Situação |
|---|---|
| Dados e features | corretos, auditados, sem vazamento e sem valores não finitos |
| Rótulo | triple barrier com θ proporcional à volatilidade — defensável |
| Split | temporal, com purging e embargo; sete anos realmente fora da amostra |
| Modelo | treina, converge, usa as três classes |
| **Sinal preditivo** | **modesto e instável**: +0,04 de F1 macro sobre o sorteio com prior, vencendo em 4 dos 7 anos; perde em acurácia para a regra trivial |
| **Vantagem por operação** | **positiva e acima dos custos** (+0,52% contra 0,164%) |
| **Resultado financeiro** | **+32% em 6,5 anos** — abaixo do buy & hold (+205%) e abaixo do CDI (+89%) |
| **Estabilidade do resultado** | **ruim**: três dias a mais de dados mudaram o retorno de +62% para +32% |

Essa tabela é o produto do notebook, e ela é mais interessante do que os dois desfechos que a
gente costuma ver por aí. Não é o caso de "não há sinal": há. Também não é o caso de "achamos
dinheiro": não achamos. É o caso, muito mais comum na prática e quase nunca publicado, de uma
**vantagem estatística real que não se converte em retorno** depois de passar pelo giro, pela
variância, pelos custos e pelo imposto.

## §7.2 — Por que esse é o desfecho típico

Direção de preço diário é **quase** um passeio aleatório, por um motivo econômico: se houvesse um
padrão simples e estável em 23 indicadores técnicos de um papel muito líquido, ele já teria sido
arbitrado. O que sobra de previsível tende a ser pequeno, instável e **caro de capturar** — e a
palavra que importa nessa frase é a última.

Nosso caso ilustra os três destinos possíveis de uma vantagem pequena:

| Destino | O que aconteceu aqui |
|---|---|
| A fricção come | R$ 4.402 de custos + IR sobre R$ 10.000 de capital |
| A variância dilui | +0,52% de média por operação rendendo muito menos que $(1{,}0052)^{132}$ |
| O custo de oportunidade supera | 63% do tempo fora de um papel que subiu 224% |
| E a medida oscila | o mesmo código, com 3 dias a mais de dados, deu metade do retorno |

Então por que treinar? Porque **você precisa saber medir**. O valor construído aqui não é a rede
— é o aparato que permite dizer, com números, *onde* a vantagem se perdeu. Com ele você pode:

- atacar o giro (operar só com alta confiança — exercício 3.2);
- trocar o alvo (volatilidade futura é bem mais previsível que direção — exercício 3.1);
- trocar o horizonte, o universo ou as features;
- recuperar o tempo fora do mercado (exercício 4.1).

E, a cada troca, medir **do mesmo jeito** — porque a medição é a parte que não pode mudar
conforme o resultado.

## §7.3 — Os erros que este notebook evitou de propósito

| Erro comum | Onde foi barrado |
|---|---|
| Embaralhar a série no `train_test_split` | split temporal, §4 do notebook anterior |
| Normalizar antes de dividir treino/teste | `TrainOnlyScaler`, ajustado só no treino |
| Rótulo desalinhado da janela | recálculo independente a partir de `raw` |
| `inf` atravessando o `dropna` | `assert` de finitude antes do scaler |
| Escolher a época pelo teste | early stopping na validação; teste tocado uma vez |
| Executar no fechamento que você acabou de ver | execução na abertura do dia seguinte |
| Backtest sem custo | emolumentos + slippage + IR, todos declarados |
| Comparar só com buy & hold | comparação com a Selic |

> 🎯 Cada linha dessa tabela é um resultado positivo que você **não** vai reportar. Metodologia
> serve para isso: a maior parte do trabalho de um bom pipeline consiste em não se enganar.

---

---

# ✅ CHECKLIST DO TREINO E DO BACKTEST

- [x] Tensores conferidos na entrada: finitos e com rótulos em `{0,1,2}`
- [x] `Conv1d` recebe `(N, F, T)` — o `transpose` está no `forward` e comentado
- [x] `forward` devolve logits; softmax só na leitura de probabilidade
- [x] Parâmetros comparados com o número de amostras antes de escolher a arquitetura
- [x] Pesos de classe na *loss*, calculados só no treino
- [x] Early stopping pela métrica de decisão (F1 macro) e **restauração** do melhor estado
- [x] Um modelo por ano, sempre com dados anteriores (walk-forward)
- [x] Comparação com quatro baselines, incluindo "sempre compra"
- [x] Preços do backtest **ajustados**, iguais aos usados nas features
- [x] Execução na abertura seguinte, nunca no fechamento da decisão
- [x] Emolumentos, slippage e IR com isenção por volume de vendas
- [x] Comparação com CDI, não só com buy & hold
- [x] Conta de equilíbrio: custo por operação × resultado médio por operação
- [x] Ticker lido da procedência dos tensores, não escrito à mão neste notebook
- [x] Resultado conferido em **dois recortes de dados** antes de ser reportado

---

# 📝 EXERCÍCIOS

### Nível 1 — medindo o que já está aí

**1.1** Troque a execução para o **fechamento do dia da decisão** (o erro comum) e compare o
retorno. Quantos pontos percentuais de lucro vêm apenas dessa fantasia?

**1.2** Zere `SLIPPAGE` e depois dobre-o para 0,10%. Monte uma tabela retorno × slippage. Em que
valor a estratégia deixa de perder?

**1.3** Rode `treinar_fold` com **cinco sementes** diferentes no mesmo fold e compare o `f1_teste`.
Calcule a média e o desvio-padrão. A dispersão entre sementes é maior ou menor que a margem de
0,04 que a rede tem sobre o sorteio com prior? (Esta é a pergunta que decide se o resultado da
Fase 5 significa algo. Suspeite da resposta antes de calcular, e depois confira.)

**1.4** Reproduza a instabilidade da §6.1: rode o backtest com o sinal truncado em
`oos.index[:-30]`, `[:-15]` e completo. Se remover 30 dias do fim mudar o retorno de 6,5 anos em
mais de 5 pontos percentuais, o que isso diz sobre a precisão com que você pode reportar esse
número?

### Nível 2 — arquitetura e treino

**2.1** Remova a `Conv1d` e deixe a LSTM ler os 30 passos crus. Depois remova a LSTM e deixe só
convolução + *pooling* global. Qual das três arquiteturas generaliza melhor — e a diferença
sobrevive ao teste de sementes de 1.3?

**2.2** Passe `(N, T, F)` direto para a `Conv1d`, sem o `transpose`. O código roda. Compare o
F1 e explique, em duas linhas, o que a rede passou a convolver.

**2.3** Substitua o último passo da LSTM (`saida[:, -1]`) por *attention pooling* sobre todos os
passos. Ganhou algo? Conte também os parâmetros novos.

### Nível 3 — o alvo e a estratégia

**3.1** Troque o alvo: em vez de direção, preveja se a **volatilidade** dos próximos 5 dias será
maior ou menor que a dos últimos 20 (duas classes). Volatilidade tem memória — o F1 deve subir
bastante. O que isso diz sobre *qual* pergunta o mercado deixa responder?

**3.2** Use a **probabilidade**, não a classe: só compre quando `prob_compra > limiar`. Varra o
limiar de 0,33 a 0,80 e monte a curva de retorno líquido × limiar. Menos operações compensam
menos acertos?

**3.3** Implemente *long/short*: `venda` vira posição vendida. Acrescente custo de aluguel de
0,02% ao dia. A simetria ajuda ou o custo come a diferença?

### Nível 4 — desafio

**4.1** Corrija o artefato do aquecimento: faça `prepare_fold` montar as janelas de teste usando
os últimos $T-1$ dias do bloco **anterior**, sem vazar rótulo. Garanta com um `assert` que
nenhuma janela de teste contém dado posterior a $t$. Quantos dias de sinal você recupera por ano?

**4.2** Rode o pipeline completo em **cinco papéis** (`VALE3`, `ITUB4`, `BBAS3`, `WEGE3`,
`KNRI11`). O F1 médio muda? Há papel em que o sinal é consistentemente melhor — e isso sobrevive
ao fato de você ter testado cinco e escolhido o melhor? (Pesquise **correção de múltiplas
comparações**.)

**4.3** Treine **um** modelo com os cinco papéis juntos (concatenando as janelas, com um
identificador de papel como feature) e compare com os cinco modelos separados. Mais dados por
modelo contra a suposição de que o padrão é o mesmo em todos: qual ganha?

**4.4** Implemente *purged k-fold cross-validation* (López de Prado) em vez do walk-forward
simples e compare a variância das estimativas de F1. Qual dos dois dá uma estimativa mais
estável — e qual é mais parecido com o uso real?

---

**Próxima aula:** de notebook a serviço — `export_model.py`, API com FastAPI e monitoramento de
*drift* em produção (o mesmo caminho que `cnn/deploy/` e `nlp1/deploy/` já percorreram).